# Competição de Carteira de Investimentos 2026 - PPGOLD

## Evolução da Estratégia: Otimização Ampla da B3

Em vez de pré-selecionar os 4 ativos manualmente, introduzimos uma abordagem estatística robusta que avalia todo o mercado listado na B3:
1. **Screening Universal (Fundamentus):** Varremos o mercado buscando alta liquidez (Faturamento > R$ 5 mi/dia), rentabilidade (ROE > 15%) e bom valuation (P/L razoável).
2. **Magic Formula Simplificada:** Classificamos as ações com base na soma do ranking de P/L (mais barato) e ROE (mais rentável) para selecionar o Top 12 da B3.
3. **Análise Combinatória e Programação Não-Linear:** Avaliamos todas as combinações possíveis de 4 ativos deste grupo de elite (495 carteiras distintas). Para cada uma, calculamos os pesos ótimos via otimizador (`scipy.optimize`) para extrair o Maior Índice de Sharpe.
A carteira final selecionada conterá rigorosamente **4 ativos**.

In [ ]:
import requests
import pandas as pd
import io
import numpy as np
import yfinance as yf
from itertools import combinations
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

## 1. Web Scraping: Varrendo todas as ações da B3 (Fundamentus)

In [ ]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
url = 'https://www.fundamentus.com.br/resultado.php'
resposta = requests.get(url, headers=headers)
df_b3 = pd.read_html(io.StringIO(resposta.text), decimal=',', thousands='.')[0]

def limpar_porcentagem(serie):
    return pd.to_numeric(serie.astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False).str.replace('%', '', regex=False), errors='coerce') / 100.0

df_b3['ROE'] = limpar_porcentagem(df_b3['ROE'])
df_b3['Liq.2meses'] = pd.to_numeric(df_b3['Liq.2meses'], errors='coerce')
df_b3['P/L'] = pd.to_numeric(df_b3['P/L'], errors='coerce')

# Exibir amostra
df_b3.head()

## 2. Modelagem Baseada em Fatores (Factor Investing / Screening)

In [ ]:
# Filtros agressivos: Apenas empresas altamente líquidas e saudáveis
filtro = df_b3[(df_b3['Liq.2meses'] > 5000000) & (df_b3['P/L'] > 0) & (df_b3['P/L'] < 30) & (df_b3['ROE'] > 0.15)].copy()

# Ranking de Valuation (P/L mais baixo = melhor)
filtro['rank_pl'] = filtro['P/L'].rank(ascending=True)
# Ranking de Rentabilidade (ROE mais alto = melhor)
filtro['rank_roe'] = filtro['ROE'].rank(ascending=False)

# Soma dos ranks (Ações baratas e rentáveis ganham)
filtro['rank_total'] = filtro['rank_pl'] + filtro['rank_roe']

# Selecionar os 12 papéis da 'nata' da bolsa
top_12_df = filtro.sort_values('rank_total').head(12)
top_12_tickers = [t + '.SA' for t in top_12_df['Papel'].tolist()]

print("Top 12 Ações da B3 pré-selecionadas pelo modelo:", top_12_tickers)

## 3. Extração dos Dados Históricos de Cotações

In [ ]:
# Puxando últimos 5 anos de todos os 12 ativos
data_inicio = '2021-03-26'
data_fim = '2026-03-26'
precos = yf.download(top_12_tickers, start=data_inicio, end=data_fim)['Close']

# Limpar ativos com muitos NaNs (empresas novas)
precos = precos.dropna(axis=1)
tickers_validos = precos.columns.tolist()

retornos_diarios = precos.pct_change().dropna()
retornos_medios_anuais = retornos_diarios.mean() * 252
covariancia_anual = retornos_diarios.cov() * 252

## 4. Otimização Combinatória da Carteira (Rigorosamente 4 Ativos)
Avaliamos cada combinação possível de 4 ativos a partir do grupo de elite. Otimizamos matematicamente o peso na sub-carteira usando `scipy.optimize` sob um risco estrutural máximo. A meta é o Índice de Sharpe.

In [ ]:
rf = 0.1075  # Taxa livre de risco (Proxy CDI)
melhor_sharpe = -100
melhor_carteira = None
pesos_otimos = None
estatisticas_finais = None

# Função objetivo para o otimizador: minimizar o Sharpe Negativo
def negative_sharpe(weights, mean_ret, cov_matrix, risk_free_rate):
    p_ret = np.sum(mean_ret * weights)
    p_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return -(p_ret - risk_free_rate) / p_vol

todas_combinacoes = list(combinations(tickers_validos, 4))

for comb in todas_combinacoes:
    ativos_sub = list(comb)
    ret_sub = retornos_medios_anuais[ativos_sub].values
    cov_sub = covariancia_anual.loc[ativos_sub, ativos_sub].values
    
    # Condições Iniciais (pesos iguais) e Restrições (soma=1, limites 0 a 1)
    num_assets = 4
    args = (ret_sub, cov_sub, rf)
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for asset in range(num_assets))
    
    resultado_opt = minimize(negative_sharpe, num_assets*[1./num_assets,], args=args,
                             method='SLSQP', bounds=bounds, constraints=constraints)
    
    sharpe_atual = -resultado_opt.fun
    if sharpe_atual > melhor_sharpe:
        melhor_sharpe = sharpe_atual
        melhor_carteira = ativos_sub
        pesos_otimos = resultado_opt.x
        ret_esperado = np.sum(ret_sub * pesos_otimos)
        vol_esperada = np.sqrt(np.dot(pesos_otimos.T, np.dot(cov_sub, pesos_otimos)))
        estatisticas_finais = (ret_esperado, vol_esperada, sharpe_atual)

print("====== RESULTADO DA OTIMIZAÇÃO (MARKOWITZ + COMBINATÓRIA) ======")
print("As 4 melhores ações selecionadas empiricamente entre TODAS da B3:")
for ativo, peso in zip(melhor_carteira, pesos_otimos):
    print(f"> {ativo}: {peso*100:.2f}%")

print(f"\nRetorno Esperado do Portfólio: {estatisticas_finais[0]*100:.2f}%")
print(f"Volatilidade (Risco): {estatisticas_finais[1]*100:.2f}%")
print(f"Índice de Sharpe Global: {melhor_sharpe:.4f}")

## 5. Próximos Passos (Aprimoramento Futuro)
Neste ponto do projeto, encontramos matematicamente a melhor alocação baseada em varredura fundamentalista e quantitativa.

**O próximo degrau será:** Incorporar uma Análise de Sentimento das Notícias em tempo real, capturando como a mídia financeira está abordando essas empresas hoje (NLP com bibliotecas como `vaderSentiment` ou via LLM de processamento) para ajustar a ponderação penalizando papéis expostos à notícias negativas recentes, refinando nosso risco subjetivo não precificado pela covariância histórica.